In [1]:
import os
import ast
import pandas as pd
import osmnx as ox

DATA_FILE = "osm_segment_metadata.csv"
GRAPH_FILE = "bhubaneswar_drive.graphml"

print("Loading traffic metadata...")
df = pd.read_csv(DATA_FILE)

print("Loading OSM graph...")
G = ox.load_graphml(GRAPH_FILE)

# ---------------------------------------------------------
# 1. Extract every individual OSM ID from dataset
# ---------------------------------------------------------

def parse_osmid(value):
    if pd.isna(value):
        return []

    value = str(value).strip()

    # List stored as string: "[123, 456, 789]"
    if value.startswith("[") and value.endswith("]"):
        try:
            parsed = ast.literal_eval(value)
            return [str(x) for x in parsed]
        except:
            return []

    # Single OSM ID
    return [value]


df["osmid_list"] = df["osmid"].apply(parse_osmid)

# ---------------------------------------------------------
# 2. Extract OSM IDs from OSMnx graph
# ---------------------------------------------------------

graph_osmids = set()

for _, _, _, data in G.edges(keys=True, data=True):

    osmid = data.get("osmid")

    if osmid is None:
        continue

    if isinstance(osmid, list):
        for oid in osmid:
            graph_osmids.add(str(oid))
    else:
        graph_osmids.add(str(osmid))

print("\n========== OSM GRAPH ==========")
print("Graph nodes:", len(G.nodes))
print("Graph edges:", len(G.edges))
print("Unique OSM IDs:", len(graph_osmids))

# ---------------------------------------------------------
# 3. Match dataset segments
# ---------------------------------------------------------

matched_segments = 0
unmatched_segments = 0

matched_ids = []
unmatched_ids = []

for _, row in df.iterrows():

    dataset_ids = row["osmid_list"]

    found = [
        oid for oid in dataset_ids
        if oid in graph_osmids
    ]

    if found:
        matched_segments += 1
        matched_ids.append(found)
    else:
        unmatched_segments += 1
        unmatched_ids.append(dataset_ids)

# ---------------------------------------------------------
# 4. Results
# ---------------------------------------------------------

total = len(df)

match_percentage = (
    matched_segments / total * 100
    if total > 0 else 0
)

print("\n========== MATCHING RESULTS ==========")
print("Total traffic segments:", total)
print("Matched segments:", matched_segments)
print("Unmatched segments:", unmatched_segments)
print(f"Match percentage: {match_percentage:.2f}%")

print("\n========== UNMATCHED SAMPLE ==========")

for ids in unmatched_ids[:20]:
    print(ids)

# ---------------------------------------------------------
# 5. Save results
# ---------------------------------------------------------

df["matched_osm_ids"] = matched_ids + [[]] * (
    len(df) - len(matched_ids)
)

df.to_csv(
    "osm_mapping_results.csv",
    index=False
)

print("\nSaved results to: osm_mapping_results.csv")

Loading traffic metadata...
Loading OSM graph...

========== OSM GRAPH ==========
Graph nodes: 18260
Graph edges: 46679
Unique OSM IDs: 11337

========== MATCHING RESULTS ==========
Total traffic segments: 700
Matched segments: 700
Unmatched segments: 0
Match percentage: 100.00%

========== UNMATCHED SAMPLE ==========

Saved results to: osm_mapping_results.csv
